In [ ]:
!pip install peft

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling
)
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm


In [ ]:
# Gendered dataset
df = pd.read_csv("data/combined_letters_gendered.csv")[["full_text", "label"]].dropna()
df["label_text"] = df["label"].map({1: "male", 0: "female"})

X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label_text"],
    test_size=0.2,
    stratify=df["label_text"],
    random_state=seed
)

def format_prompt(text, label=None):
    prompt = (
        f"Based on the content of the following letter of recommendation, determine if the applicant is male or female:\n"
        f"\"{text}\"\nAnswer:"
    )
    return prompt if label is None else f"{prompt} {label}"

train_prompts = [format_prompt(txt, label) for txt, label in zip(X_train, y_train)]
test_prompts = [format_prompt(txt) for txt in X_test]


In [ ]:
# Model
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


In [ ]:
# LORA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)


In [ ]:
# Prepare datasets
train_dataset = Dataset.from_dict({"text": train_prompts})
test_dataset = Dataset.from_dict({"text": test_prompts, "label": y_test.tolist()})

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results_llama2_gendered",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="no",
    logging_steps=10,
    report_to=None,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [ ]:
# Training
trainer.train()

In [ ]:
# Inference
def classify(prompt, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("Answer:")[-1].strip().lower()

predictions = [classify(p) for p in tqdm(test_prompts)]

In [ ]:
# Evaluation
def compute_metrics(true, pred):
    pred_clean = [p if p in {"male", "female"} else "unknown" for p in pred]
    true_clean = [t.lower() for t in true]

    acc = accuracy_score(true_clean, pred_clean)
    precision, recall, f1, _ = precision_recall_fscore_support(true_clean, pred_clean, average="macro", zero_division=0)
    mcc = matthews_corrcoef(true_clean, pred_clean)
    bal_acc = balanced_accuracy_score(true_clean, pred_clean)
    kappa = cohen_kappa_score(true_clean, pred_clean)
    jaccard = jaccard_score(true_clean, pred_clean, average="macro", labels=["male", "female"])
    hamming = hamming_loss(true_clean, pred_clean)
    cm = confusion_matrix(true_clean, pred_clean, labels=["male", "female", "unknown"])

    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print("Confusion Matrix:\n", cm)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "cohen_kappa": kappa,
        "jaccard": jaccard,
        "hamming_loss": hamming
    }

metrics = compute_metrics(y_test.tolist(), predictions)
